# 02 · Engine checks

Before any experiment, this notebook checks that the engines behind it are correct. The engines are plain Python files, so every notebook uses exactly the same code:

| File | What it does |
|---|---|
| `quantum_kernel.py` | encodes each row into a quantum state and computes the fidelity kernel |
| `classical_models.py` | the three classical baselines and their tuning grids |
| `corruption.py` | missing values, measurement noise and label flips (checked in section 3) |

All checks use the 280 training rows of each dataset; the test rows are not touched. Nothing is saved.

## 0 · Setup

In [7]:
import time
import numpy as np
import pandas as pd
from quantum_kernel import load_ready, get_states, get_states_pennylane, kernel_from_states, off_diagonal_stats
from classical_models import get_model, SEEDS, SCORING

pd.set_option("display.width", 200)

DATASETS = ["german_credit", "concrete", "water_potability", "steel_faults",
            "phoneme", "magic_telescope", "bank_marketing", "electricity"]
BANDWIDTHS = [0.01, 0.025, 0.05, 0.1, 0.25, 0.5, 0.75, 1.0]

data = {name: load_ready(name)[:2] for name in DATASETS}
pd.DataFrame({name: {"qubits": X.shape[1], "training rows": len(X)} for name, (X, y) in data.items()}).T

,qubits,training rows
german_credit,14,280
concrete,8,280
water_potability,9,280
steel_faults,14,280
phoneme,5,280
magic_telescope,8,280
bank_marketing,14,280
electricity,6,280


## 1 · Quantum kernel

Each row is encoded into a quantum state by the ZZ feature map: on every qubit a Hadamard gate and a rotation by the feature value, then an entangling rotation between neighbouring qubits; the whole block is repeated twice. Every feature is first multiplied by the **bandwidth c**:

- small c ("zoomed out") keeps the rotations small, so all states stay similar;
- large c ("zoomed in") makes every state different from every other.

The kernel is the **fidelity** between two states, |⟨a|b⟩|², a similarity between 0 and 1. The SVM only needs this table of similarities (the Gram matrix).

### Check 1 · The fast encoder builds the same circuit

`get_states` writes the circuit as matrix algebra, which is much faster than building it gate by gate. It is compared with the gate-by-gate PennyLane version on 10 rows of every dataset, at a small and a large bandwidth. Differences of about 1e-14 are rounding error, so the two are the same circuit.

In [8]:
rows = {}
for name, (X, y) in data.items():
    q, Xs = X.shape[1], X[:10]
    start = time.perf_counter()
    diff_states, diff_kernel = 0.0, 0.0
    for c in [0.05, 1.0]:
        fast, ref = get_states(Xs, q, c=c), get_states_pennylane(Xs, q, c=c)
        diff_states = max(diff_states, np.abs(fast - ref).max())
        diff_kernel = max(diff_kernel, np.abs(kernel_from_states(fast, fast) - kernel_from_states(ref, ref)).max())
    rows[name] = {"qubits": q, "largest state difference": f"{diff_states:.1e}",
                  "largest kernel difference": f"{diff_kernel:.1e}",
                  "same circuit": diff_kernel < 1e-10, "seconds": round(time.perf_counter() - start, 1)}

pd.DataFrame(rows).T

,qubits,largest state difference,largest kernel difference,same circuit,seconds
german_credit,14,2.7e-14,1.1e-14,True,0.5
concrete,8,1.4e-14,8.7e-15,True,0.2
water_potability,9,1.1e-14,8.3e-15,True,0.2
steel_faults,14,1.7e-14,1.2e-14,True,0.3
phoneme,5,4.7e-15,4.7e-15,True,0.1
magic_telescope,8,1.0e-14,8.7e-15,True,0.1
bank_marketing,14,2.0e-14,1.2e-14,True,0.5
electricity,6,5.9e-15,6.4e-15,True,0.1


### Check 2 · Every Gram matrix is a valid kernel

On all 280 training rows and all eight bandwidths, a valid Gram matrix must have ones on the diagonal (every row is identical to itself), be symmetric, and have no negative eigenvalues beyond rounding error.

In [9]:
grams, rows = {}, {}
for name, (X, y) in data.items():
    worst = {"diagonal": 0.0, "symmetry": 0.0, "smallest eigenvalue": np.inf}
    for c in BANDWIDTHS:
        S = get_states(X, X.shape[1], c=c)
        K = kernel_from_states(S, S)
        grams[name, c] = K
        worst["diagonal"] = max(worst["diagonal"], np.abs(np.diag(K) - 1).max())
        worst["symmetry"] = max(worst["symmetry"], np.abs(K - K.T).max())
        worst["smallest eigenvalue"] = min(worst["smallest eigenvalue"], np.linalg.eigvalsh(K).min())
    rows[name] = {"largest |diagonal - 1|": f"{worst['diagonal']:.1e}", "largest |K - Kᵀ|": f"{worst['symmetry']:.1e}",
                  "smallest eigenvalue": f"{worst['smallest eigenvalue']:.1e}",
                  "valid at all 8 c": worst["diagonal"] < 1e-10 and worst["symmetry"] < 1e-10
                                      and worst["smallest eigenvalue"] > -1e-10}

pd.DataFrame(rows).T

,largest |diagonal - 1|,largest |K - Kᵀ|,smallest eigenvalue,valid at all 8 c
german_credit,6.2e-15,3.9e-15,2.0e-11,True
concrete,3.3e-15,3.7e-15,-5.1e-15,True
water_potability,4.7e-15,5.1e-15,9.6e-12,True
steel_faults,4.2e-15,5.1e-15,1.2e-09,True
phoneme,2.4e-15,1.9e-15,-4.5e-14,True
magic_telescope,4.4e-15,3.7e-15,4.7e-16,True
bank_marketing,4.0e-15,3.8e-15,2.9e-09,True
electricity,3.6e-15,2.0e-15,-4.8e-15,True


### Check 3 · The concentration meter

The spread (variance) of the off-diagonal similarities shows how much the kernel can tell rows apart. When it falls towards 0, all pairs look equally (dis)similar: the kernel has **concentrated**. The table shows the expected pattern: the spread is largest at a middle bandwidth and collapses at large c, fastest for the 14-qubit datasets.

In [10]:
spread = pd.DataFrame({name: {c: off_diagonal_stats(grams[name, c])[1] for c in BANDWIDTHS} for name in DATASETS}).T
spread.columns = [f"c = {c:g}" for c in BANDWIDTHS]
spread.insert(0, "qubits", [data[n][0].shape[1] for n in DATASETS])
spread.style.format({col: "{:.1e}" for col in spread.columns[1:]}).background_gradient(
    cmap="Blues", subset=spread.columns[1:], axis=None, gmap=np.log10(spread.iloc[:, 1:].clip(lower=1e-12)))

,qubits,c = 0.01,c = 0.025,c = 0.05,c = 0.1,c = 0.25,c = 0.5,c = 0.75,c = 1
german_credit,14,1.6e-03,1.9e-02,3.2e-02,9.5e-03,9.2e-04,2.9e-04,1.6e-04,7.0e-04
concrete,8,1.4e-04,3.1e-03,1.8e-02,5.1e-02,1.1e-02,3.3e-03,2.5e-03,2.0e-03
water_potability,9,7.6e-05,1.6e-03,1.1e-02,4.0e-02,4.0e-03,3.1e-04,6.4e-05,4.5e-05
steel_faults,14,1.2e-03,1.6e-02,3.4e-02,1.5e-02,1.3e-03,1.9e-04,5.3e-05,3.3e-05
phoneme,5,5.4e-05,1.3e-03,1.1e-02,6.2e-02,2.7e-02,1.1e-02,6.3e-03,5.3e-03
magic_telescope,8,2.2e-04,4.6e-03,2.7e-02,8.5e-02,3.0e-02,5.4e-03,2.0e-03,1.0e-03
bank_marketing,14,1.1e-03,1.5e-02,3.2e-02,1.1e-02,5.3e-04,1.0e-04,4.7e-05,2.7e-04
electricity,6,2.8e-04,5.5e-03,2.4e-02,6.1e-02,1.1e-02,2.5e-03,1.5e-03,1.6e-03


## 2 · Classical models

Three classical models are compared with the quantum kernel. SVM-RBF is the main comparison, because it is also a kernel method with a bandwidth (gamma); Random Forest and Logistic Regression are reference points. Every model, quantum included, is tuned by an inner 3-fold cross-validation on ROC-AUC, uses balanced class weights, and is scored on the same 5 seeds × 5 outer folds.

In [11]:
pd.DataFrame([{"model": name, "estimator": type(get_model(name)[0]).__name__,
               "tuned in the inner loop": ", ".join(f"{k} = {v}" for k, v in get_model(name)[1].items())}
              for name in ["svm_rbf", "random_forest", "logistic_regression"]]
             + [{"model": "quantum kernel", "estimator": "SVC (precomputed kernel)",
                 "tuned in the inner loop": f"c = {BANDWIDTHS}, C = [0.1, 1, 10]"}]).set_index("model")

,estimator,tuned in the inner loop
model,,
svm_rbf,SVC,"gamma = [0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0..."
random_forest,RandomForestClassifier,"max_depth = [None, 5, 10], min_samples_leaf = ..."
logistic_regression,LogisticRegression,"C = [0.01, 0.1, 1, 10]"
quantum kernel,SVC (precomputed kernel),"c = [0.01, 0.025, 0.05, 0.1, 0.25, 0.5, 0.75, ..."


In [12]:
print("seeds:", SEEDS, "| outer folds: 5 | inner folds: 3 | metrics:", ", ".join(SCORING.values()))

seeds: [0, 1, 2, 3, 4] | outer folds: 5 | inner folds: 3 | metrics: accuracy, f1_macro, roc_auc


## 3 · Corruption engine

Three kinds of damage are applied to the **training rows only**, each at three levels. The levels are **nested**: the cells or labels damaged at a low level are also damaged at every higher level, so levels differ only by the extra damage.

| Damage | Levels | What happens |
|---|---|---|
| Missing values | 10, 20, 30 % of cells | cells hidden completely at random, then filled with the median of the same (damaged) training rows |
| Measurement noise | 0.05, 0.15, 0.30 × feature SD | Gaussian error added to **measured columns only** |
| Wrong labels | 5, 10, 15 % of rows | labels flipped at random, in both directions |

Noise is a measurement error, so it is added only to measured quantities (amounts, sizes, durations, ages, counts). Category codes, yes/no flags, ordered levels and calendar positions keep their values. The checks below use the 280 training rows of each dataset with one fixed seed.

### Which columns receive noise

In [13]:
from corruption import (LEVELS, corruption_seed, corrupt_training, add_mcar, add_gaussian_noise,
                        flip_labels, mcar_then_impute, noise_columns, feature_names)

noise_cols = {name: noise_columns(name) for name in DATASETS}
pd.DataFrame({name: {"qubits": len(cols), "columns with noise": int(cols.sum()),
                     "columns without noise (codes)": ", ".join(np.array(feature_names(name))[~cols]) or "-"}
              for name, cols in noise_cols.items()}).T

,qubits,columns with noise,columns without noise (codes)
german_credit,14,3,"f4, f2, f0, f11, f21, f10, f15, f22, f13, f18,..."
concrete,8,7,Age (day)
water_potability,9,9,-
steel_faults,14,12,"Outside_Global_Index, TypeOfSteel_A400"
phoneme,5,5,-
magic_telescope,8,8,-
bank_marketing,14,4,"job, marital, education, default, housing, loa..."
electricity,6,4,"day, period"


### Check 4 · Missing values

The exact number of cells is hidden at each level, the hidden cells are nested, and after filling no gap is left.

In [14]:
rows = {}
for name, (X, y) in data.items():
    seed = corruption_seed(name, "missing", 0, 0)
    masks = [add_mcar(X, rate, seed)[1] for rate in LEVELS["missing"]]
    filled = [mcar_then_impute(X, rate, seed)[0] for rate in LEVELS["missing"]]
    rows[name] = {"cells": X.size,
                  "hidden (10 / 20 / 30 %)": " / ".join(str(m.sum()) for m in masks),
                  "exact counts": all(m.sum() == round(r * X.size) for m, r in zip(masks, LEVELS["missing"])),
                  "nested": bool((masks[0] <= masks[1]).all() and (masks[1] <= masks[2]).all()),
                  "gaps left after filling": int(sum(np.isnan(f).sum() for f in filled))}

pd.DataFrame(rows).T

,cells,hidden (10 / 20 / 30 %),exact counts,nested,gaps left after filling
german_credit,3920,392 / 784 / 1176,True,True,0
concrete,2240,224 / 448 / 672,True,True,0
water_potability,2520,252 / 504 / 756,True,True,0
steel_faults,3920,392 / 784 / 1176,True,True,0
phoneme,1400,140 / 280 / 420,True,True,0
magic_telescope,2240,224 / 448 / 672,True,True,0
bank_marketing,3920,392 / 784 / 1176,True,True,0
electricity,1680,168 / 336 / 504,True,True,0


### Check 5 · Measurement noise

Code columns must stay exactly as they were; measured columns must receive noise of the intended size (ratio ≈ 1). The noise is one random draw scaled up, so the 0.30 noise is exactly twice the 0.15 noise.

In [15]:
rows = {}
for name, (X, y) in data.items():
    cols, seed = noise_cols[name], corruption_seed(name, "noise", 0, 0)
    noise = {lv: add_gaussian_noise(X, lv, seed, cols)[1] for lv in LEVELS["noise"]}
    sd = X.std(axis=0, ddof=1)
    ratio = (noise[0.15][:, cols].std(axis=0, ddof=1) / (0.15 * sd[cols])).mean() if cols.any() else np.nan
    rows[name] = {"code columns unchanged": bool((noise[0.30][:, ~cols] == 0).all()),
                  "noise size / intended (0.15)": round(ratio, 3),
                  "nested (0.30 = 2 × 0.15)": bool(np.allclose(noise[0.30], 2 * noise[0.15])),
                  "values outside [0, π] at 0.30": f"{((X + noise[0.30] < 0) | (X + noise[0.30] > np.pi)).mean():.1%}"}

pd.DataFrame(rows).T

,code columns unchanged,noise size / intended (0.15),nested (0.30 = 2 × 0.15),"values outside [0, π] at 0.30"
german_credit,True,0.982,True,1.0%
concrete,True,1.01,True,9.3%
water_potability,True,1.0,True,0.7%
steel_faults,True,0.99,True,11.4%
phoneme,True,0.984,True,1.9%
magic_telescope,True,1.004,True,5.2%
bank_marketing,True,1.019,True,4.7%
electricity,True,1.015,True,0.8%


### Check 6 · Wrong labels

Exactly 14, 28 and 42 of the 280 labels are flipped, the flipped rows are nested, and the features are untouched.

In [16]:
rows = {}
for name, (X, y) in data.items():
    seed = corruption_seed(name, "labels", 0, 0)
    flipped = [flip_labels(y, rate, seed)[1] for rate in LEVELS["labels"]]
    X_c, y_c = corrupt_training(X, y, "labels", 0.15, seed)
    rows[name] = {"flipped (5 / 10 / 15 %)": " / ".join(str(len(f)) for f in flipped),
                  "nested": set(flipped[0]) <= set(flipped[1]) <= set(flipped[2]),
                  "features unchanged": bool(np.array_equal(X_c, X)),
                  "label = 1 before → after 15 %": f"{y.mean():.0%} → {y_c.mean():.0%}"}

pd.DataFrame(rows).T

,flipped (5 / 10 / 15 %),nested,features unchanged,label = 1 before → after 15 %
german_credit,14 / 28 / 42,True,True,70% → 66%
concrete,14 / 28 / 42,True,True,50% → 48%
water_potability,14 / 28 / 42,True,True,39% → 45%
steel_faults,14 / 28 / 42,True,True,35% → 41%
phoneme,14 / 28 / 42,True,True,29% → 36%
magic_telescope,14 / 28 / 42,True,True,35% → 34%
bank_marketing,14 / 28 / 42,True,True,12% → 25%
electricity,14 / 28 / 42,True,True,42% → 42%


### Check 7 · Safe to use

The same seed always gives the same damage, and the original training rows are never changed.

In [ ]:
X, y = data["german_credit"]
X0, y0 = X.copy(), y.copy()
same = all(np.array_equal(corrupt_training(X, y, kind, LEVELS[kind][-1], 7, noise_cols["german_credit"])[i],
                          corrupt_training(X, y, kind, LEVELS[kind][-1], 7, noise_cols["german_credit"])[i])
           for kind in LEVELS for i in (0, 1))
print("same seed, same result:", same)
print("input rows unchanged:  ", np.array_equal(X, X0) and np.array_equal(y, y0))

same seed, same result: True
input rows unchanged:   True


: 